# 04. Construcción de los conjuntos de entrenamiento, validación y prueba

En este cuaderno realizo la separación temporal del conjunto de características generado previamente.

Mi objetivo es construir los subconjuntos que utilizaré durante el entrenamiento, ajuste y evaluación final de los modelos predictivos. Debido a que trabajo con información deportiva ordenada cronológicamente, no realizo una división aleatoria de los registros.

Utilizo los periodos históricos más antiguos para entrenamiento, un periodo posterior para validación y los encuentros más recientes como conjunto de prueba final. Con esta estrategia reproduzco de forma más realista el escenario en el que utilizo información pasada para predecir partidos futuros y reduzco el riesgo de fuga de información entre los subconjuntos.

# Celda 2 Librerías

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 100)

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


# Celda 3 Carga del conjunto de características 
## 1. Carga del conjunto de características
Cargo el archivo generado durante la etapa de ingeniería de características. Este conjunto contiene una observación por partido, las variables prepartido construidas y las variables objetivo que utilizaré durante el entrenamiento y la evaluación.

In [2]:
ROOT = Path("..").resolve()

PROCESSED_DIR = (
    ROOT
    / "data"
    / "processed"
)

ARCHIVO_FEATURES = (
    PROCESSED_DIR
    / "lpf_features_modelado.csv"
)

df_modelado = pd.read_csv(
    ARCHIVO_FEATURES,
    encoding="utf-8-sig",
    parse_dates=["fecha_partido"]
)

df_modelado = (
    df_modelado
    .sort_values(
        [
            "fecha_partido",
            "registro_id"
        ]
    )
    .reset_index(drop=True)
)

print(
    "Dimensiones:",
    df_modelado.shape
)

print(
    "Fecha inicial:",
    df_modelado["fecha_partido"].min()
)

print(
    "Fecha final:",
    df_modelado["fecha_partido"].max()
)

print(
    "Periodos disponibles:"
)

print(
    df_modelado[
        "temporada"
    ].value_counts().sort_index()
)

Dimensiones: (2226, 55)
Fecha inicial: 2014-07-18 00:00:00
Fecha final: 2026-09-14 00:00:00
Periodos disponibles:
temporada
2014/2015    190
2015/2016    190
2016/2017    190
2017/2018    189
2018/2019    194
2020          92
2021         206
2022         206
2023         206
2024         206
2025         206
2026         103
2026/2027     48
Name: count, dtype: int64


# Celda 4. Definición de la estrategia de partición temporal
# 2. Definición de la estrategia de partición temporal
Divido el conjunto de datos respetando completamente el orden temporal de la competición.
Utilizo para entrenamiento los encuentros comprendidos desde el inicio del histórico hasta el periodo 2024. Reservo los partidos de 2025 como conjunto de validación para la comparación de configuraciones e hiperparámetros. Finalmente, utilizo los registros de 2026 y 2026/2027 como conjunto de prueba final. Mantendré este último subconjunto sin utilizar durante el ajuste de los modelos para obtener posteriormente una evaluación sobre información cronológicamente posterior.

En esta celda estructuro el filtrado por temporadas mediante máscaras booleanas (mask_train, mask_validacion, mask_test) sobre la columna temporada. De esta manera se crean tres subconjuntos completamente aislados (df_train, df_validacion, df_test), lo que impide de raíz cualquier fuga temporal de datos durante la optimización de hiperparámetros.

In [5]:
# definir las listas de temporadas asociadas a cada particion cronologica
TEMPORADAS_TRAIN = [
    "2014/2015",
    "2015/2016",
    "2016/2017",
    "2017/2018",
    "2018/2019",
    "2020",
    "2021",
    "2022",
    "2023",
    "2024"
]

TEMPORADAS_VALIDACION = [
    "2025"
]

TEMPORADAS_TEST = [
    "2026",
    "2026/2027"
]

# construir las mascaras booleanas de seleccion por temporada
mask_train = (
    df_modelado[
        "temporada"
    ].isin(TEMPORADAS_TRAIN)
)

mask_validacion = (
    df_modelado[
        "temporada"
    ].isin(TEMPORADAS_VALIDACION)
)

mask_test = (
    df_modelado[
        "temporada"
    ].isin(TEMPORADAS_TEST)
)

# segmentar los DataFrames asegurando copias independientes e indices limpios
df_train = (
    df_modelado[
        mask_train
    ]
    .copy()
    .reset_index(drop=True)
)

df_validacion = (
    df_modelado[
        mask_validacion
    ]
    .copy()
    .reset_index(drop=True)
)

df_test = (
    df_modelado[
        mask_test
    ]
    .copy()
    .reset_index(drop=True)
)

# Celda 5 Tamaño de cada subconjunto

In [6]:
resumen_split = pd.DataFrame({
    "conjunto": [
        "Entrenamiento",
        "Validación",
        "Prueba"
    ],
    "partidos": [
        len(df_train),
        len(df_validacion),
        len(df_test)
    ]
})

resumen_split[
    "porcentaje"
] = (
    resumen_split[
        "partidos"
    ]
    / len(df_modelado)
    * 100
).round(2)

display(resumen_split)

print(
    "Total recuperado:",
    resumen_split[
        "partidos"
    ].sum()
)

,conjunto,partidos,porcentaje
0,Entrenamiento,1869,83.96
1,Validación,206,9.25
2,Prueba,151,6.78


Total recuperado: 2226


# Celda 6. Verificación del aislamiento temporal mediante aserciones
# 2.1. Verificación del aislamiento temporal
Compruebo que las fechas de entrenamiento sean anteriores a las de validación y que, a su vez, las observaciones utilizadas para prueba correspondan a los periodos más recientes. Esta validación me permite comprobar que ningún partido futuro haya sido incorporado dentro del conjunto utilizado para entrenar los modelos.

En esta celda estructuro un DataFrame fechas_split para contrastar los rangos de fechas (fecha_inicial y fecha_final) de df_train, df_validacion y df_test. Evalúo mediante desigualdades estrictas y sentencias assert que la fecha máxima de entrenamiento sea menor a la inicial de validación (orden_train_val), y que la máxima de validación sea menor a la inicial de prueba (orden_val_test), garantizando un esquema estricto de validación fuera de tiempo (out-of-time validation).

In [8]:
# construir la tabla resumen con los rangos temporales de cada partición
fechas_split = pd.DataFrame({
    "conjunto": [
        "Entrenamiento",
        "Validación",
        "Prueba"
    ],
    "fecha_inicial": [
        df_train["fecha_partido"].min(),
        df_validacion["fecha_partido"].min(),
        df_test["fecha_partido"].min()
    ],
    "fecha_final": [
        df_train["fecha_partido"].max(),
        df_validacion["fecha_partido"].max(),
        df_test["fecha_partido"].max()
    ]
})

# desplegar la tabla de verificación temporal
display(fechas_split)

# evaluar el orden cronológico entre entrenamiento y validación
orden_train_val = (
    df_train["fecha_partido"].max()
    <
    df_validacion["fecha_partido"].min()
)

# evaluar el orden cronológico entre validación y prueba
orden_val_test = (
    df_validacion["fecha_partido"].max()
    <
    df_test["fecha_partido"].min()
)

# mostrar el resultado de los controles booleanos
print(
    "Entrenamiento anterior a validación:",
    orden_train_val
)

print(
    "Validación anterior a prueba:",
    orden_val_test
)

# validar formalmente la secuencia cronológica
assert orden_train_val
assert orden_val_test

,conjunto,fecha_inicial,fecha_final
0,Entrenamiento,2014-07-18,2024-11-30
1,Validación,2025-01-17,2025-11-29
2,Prueba,2026-01-16,2026-09-14


Entrenamiento anterior a validación: True
Validación anterior a prueba: True


# Celda 7 Control de registros

In [9]:
ids_train = set(
    df_train["registro_id"]
)

ids_validacion = set(
    df_validacion["registro_id"]
)

ids_test = set(
    df_test["registro_id"]
)

print(
    "Train ∩ Validación:",
    len(
        ids_train
        & ids_validacion
    )
)

print(
    "Train ∩ Prueba:",
    len(
        ids_train
        & ids_test
    )
)

print(
    "Validación ∩ Prueba:",
    len(
        ids_validacion
        & ids_test
    )
)

Train ∩ Validación: 0
Train ∩ Prueba: 0
Validación ∩ Prueba: 0


# Celda 8. Análisis de la distribución de la variable objetivo por subconjunto
# 3. Distribución de la variable objetivo por subconjunto
Analizo la distribución de victorias locales, empates y victorias visitantes dentro de cada subconjunto.
No utilizo esta información para modificar artificialmente el conjunto de prueba. La revisión tiene únicamente el propósito de conocer si existen cambios temporales en la distribución de la variable objetivo.

In [10]:
# función auxiliar para calcular la distribución de frecuencias de la variable objetivo
def distribucion_objetivo(df, nombre):
    conteo = (
        df["resultado"]
        .value_counts()
        .reindex(
            ["L", "E", "V"],
            fill_value=0
        )
    )

    porcentaje = (
        df["resultado"]
        .value_counts(
            normalize=True
        )
        .reindex(
            ["L", "E", "V"],
            fill_value=0
        )
        * 100
    )

    return pd.DataFrame({
        "conjunto": nombre,
        "resultado": conteo.index,
        "partidos": conteo.values,
        "porcentaje": porcentaje.values
    })


# consolidar las distribuciones de los tres subconjuntos temporales
tabla_distribucion = pd.concat(
    [
        distribucion_objetivo(
            df_train,
            "Entrenamiento"
        ),
        distribucion_objetivo(
            df_validacion,
            "Validación"
        ),
        distribucion_objetivo(
            df_test,
            "Prueba"
        )
    ],
    ignore_index=True
)

# redondear los porcentajes a dos decimales
tabla_distribucion[
    "porcentaje"
] = tabla_distribucion[
    "porcentaje"
].round(2)

# desplegar la tabla comparativa
display(tabla_distribucion)

,conjunto,resultado,partidos,porcentaje
0,Entrenamiento,L,730,39.06
1,Entrenamiento,E,573,30.66
2,Entrenamiento,V,566,30.28
3,Validación,L,75,36.41
4,Validación,E,67,32.52
5,Validación,V,64,31.07
6,Prueba,L,56,37.09
7,Prueba,E,59,39.07
8,Prueba,V,36,23.84


# Celda 9. Detección de equipos no observados en la partición de entrenamiento 
# 4. Verificación de equipos no observados durante el entrenamiento
Compruebo si los periodos de validación o prueba contienen clubes que no aparecen dentro del conjunto de entrenamiento.
Este escenario es relevante para la capacidad de generalización del sistema. Un equipo de incorporación reciente puede carecer de una categoría conocida durante el ajuste del modelo, por lo que posteriormente configuraré la codificación categórica para tolerar categorías no observadas.


In [11]:
# función para consolidar el universo de equipos de un subconjunto
def obtener_equipos(df):
    return (
        set(
            df["home_team_name"]
        )
        |
        set(
            df["away_team_name"]
        )
    )


# identificar los conjuntos de equipos por cada partición
equipos_train = obtener_equipos(
    df_train
)

equipos_validacion = obtener_equipos(
    df_validacion
)

equipos_test = obtener_equipos(
    df_test
)

# calcular diferencias de conjuntos respecto a entrenamiento
nuevos_validacion = (
    equipos_validacion
    - equipos_train
)

nuevos_test = (
    equipos_test
    - equipos_train
)

# mostrar métricas de equipos no vistos previamente
print(
    "Equipos diferentes en entrenamiento:",
    len(equipos_train)
)

print(
    "\nEquipos de validación no vistos en entrenamiento:"
)

print(
    sorted(nuevos_validacion)
)

print(
    "\nEquipos de prueba no vistos en entrenamiento:"
)

print(
    sorted(nuevos_test)
)

Equipos diferentes en entrenamiento: 18

Equipos de validación no vistos en entrenamiento:
['Costa del Este Academia']

Equipos de prueba no vistos en entrenamiento:
['Unión Coclé']


# Celda 10. Separación formal de matrices de características ($X$) y vector objetivo ($y$)
# 5. Separación entre variables predictoras y variable objetivo
Construyo las matrices de entrada (X) y los vectores objetivo (y) manteniendo fuera de las variables predictoras cualquier información correspondiente al resultado final del encuentro.Conservo la fecha, temporada e identificador únicamente como metadatos para facilitar posteriormente la trazabilidad de las predicciones.En esta celda defino explícitamente el aislamiento entre metadatos/objetivos (columnas_no_predictoras), atributos categóricos (columnas_categoricas) y predictores cuantitativos derivados (columnas_numericas). Al consolidar columnas_X, aseguro que las matrices de características contengan únicamente la información conocida antes de la disputa del partido.

In [12]:
# definir variables excluidas del espacio de entrada (metadatos y targets)
columnas_no_predictoras = [
    "registro_id",
    "fecha_partido",
    "temporada",

    "home_team_goal_count",
    "away_team_goal_count",
    "total_goal_count",
    "resultado"
]

# definir atributos categoricos y auxiliares temporales
columnas_categoricas = [
    "home_team_name",
    "away_team_name",
    "mes_partido",
    "dia_semana"
]

# identificar dinamicamente las variables cuantitativas predictoras
columnas_numericas = [
    columna
    for columna
    in df_modelado.columns
    if (
        columna
        not in columnas_no_predictoras
        and columna
        not in columnas_categoricas
    )
]

# consolidar la lista completa de predictores candidatos (X)
columnas_X = (
    columnas_numericas
    + columnas_categoricas
)

# mostrar el desglose de predictores para modelado
print(
    "Variables numéricas:",
    len(columnas_numericas)
)

print(
    "Variables categóricas:",
    len(columnas_categoricas)
)

print(
    "Total de predictores:",
    len(columnas_X)
)

Variables numéricas: 44
Variables categóricas: 4
Total de predictores: 48


# Celda 11 Construcción de matrices

In [13]:
X_train = (
    df_train[
        columnas_X
    ].copy()
)

y_train = (
    df_train[
        "resultado"
    ].copy()
)


X_validacion = (
    df_validacion[
        columnas_X
    ].copy()
)

y_validacion = (
    df_validacion[
        "resultado"
    ].copy()
)


X_test = (
    df_test[
        columnas_X
    ].copy()
)

y_test = (
    df_test[
        "resultado"
    ].copy()
)

print(
    "X_train:",
    X_train.shape,
    "| y_train:",
    y_train.shape
)

print(
    "X_validación:",
    X_validacion.shape,
    "| y_validación:",
    y_validacion.shape
)

print(
    "X_test:",
    X_test.shape,
    "| y_test:",
    y_test.shape
)

X_train: (1869, 48) | y_train: (1869,)
X_validación: (206, 48) | y_validación: (206,)
X_test: (151, 48) | y_test: (151,)


# Celda 12. Inspección y cuantificación de valores faltantes por subconjunto 
# 5.1. Cuantificación de valores faltantes pre-imputación
Antes de aplicar el pipeline de transformaciones y modelado, audito la presencia de valores nulos dentro de cada una de las matrices de entradas .En esta celda estructuro un cuadro comparativo donde contabilizo el volumen total de celdas faltantes (.isna().sum().sum()) y el número de observaciones/filas que contienen al menos un valor nulo (.isna().any(axis=1).sum()). Esta verificación confirma la necesidad de integrar un transformador de imputación (como SimpleImputer con mediana) dentro de nuestro pipeline scikit-learn, garantizando que los parámetros de imputación se calculen estrictamente para evitar data leakage.

In [14]:
# calcular el resumen de valores nulos por partición de modelado
resumen_nan = pd.DataFrame({
    "conjunto": [
        "Entrenamiento",
        "Validación",
        "Prueba"
    ],

    "valores_faltantes": [
        X_train.isna().sum().sum(),
        X_validacion.isna().sum().sum(),
        X_test.isna().sum().sum()
    ],

    "filas_con_faltantes": [
        X_train.isna().any(axis=1).sum(),
        X_validacion.isna().any(axis=1).sum(),
        X_test.isna().any(axis=1).sum()
    ]
})

# desplegar la tabla de auditoría de nulos por subconjunto
display(resumen_nan)

,conjunto,valores_faltantes,filas_con_faltantes
0,Entrenamiento,726,67
1,Validación,67,8
2,Prueba,105,13


# Celda 13. Determinación y evaluación de la línea base 
# 6. Línea base utilizando únicamente el conjunto de entrenamiento
Determino la clase mayoritaria utilizando exclusivamente el conjunto de entrenamiento. Evalúo esa estrategia trivial sobre validación y prueba. De esta manera evito utilizar información del conjunto de prueba para decidir cuál es la clase de referencia.

In [15]:
# identificar la clase mayoritaria dentro del subconjunto de entrenamiento
clase_baseline = (
    y_train
    .value_counts()
    .idxmax()
)

print(
    "Clase mayoritaria en entrenamiento:",
    clase_baseline
)

# evaluar el desempeño del modelo ingenuo en cada subconjunto
for nombre, y in [
    ("Entrenamiento", y_train),
    ("Validación", y_validacion),
    ("Prueba", y_test)
]:
    accuracy = (
        y.eq(
            clase_baseline
        ).mean()
        * 100
    )

    print(
        f"{nombre}: "
        f"{accuracy:.2f}%"
    )

Clase mayoritaria en entrenamiento: L
Entrenamiento: 39.06%
Validación: 36.41%
Prueba: 37.09%


# Celda 14. Auditoría de clubes nuevos por subconjunto 
6.1. Presencia de equipos no observados previamente
La separación temporal identificó equipos que aparecen por primera vez después del periodo utilizado para entrenamiento. Considero este comportamiento relevante porque reproduce una situación posible durante el uso real del sistema, donde pueden incorporarse nuevos clubes a la competición. No elimino estos partidos. onfiguraré la codificación de variables categóricas para aceptar equipos no observados durante el ajuste, mientras que las características numéricas históricas permitirán representar el rendimiento disponible cuando exista información previa.

En esta celda consolido la auditoría de equipos en la tabla resumen_equipos_split, registrando la cantidad total de clubes y la presencia de equipos no observados en entrenamiento (equipos_no_vistos_train). La impresión detallada de estos nombres confirma la necesidad de manejar de forma robusta las categorías desconocidas durante el pipeline de modelado.

In [17]:
# estructurar el resumen de equipos observados por cada subconjunto
resumen_equipos_split = pd.DataFrame({
    "conjunto": [
        "Entrenamiento",
        "Validación",
        "Prueba"
    ],
    "equipos": [
        len(equipos_train),
        len(equipos_validacion),
        len(equipos_test)
    ],
    "equipos_no_vistos_train": [
        0,
        len(nuevos_validacion),
        len(nuevos_test)
    ]
})

# desplegar el resumen consolidado de auditoría de clubes
display(resumen_equipos_split)

# mostrar el listado explicativo de clubes no vistos en entrenamiento
print(
    "Nuevos en validación:",
    sorted(nuevos_validacion)
)

print(
    "Nuevos en prueba:",
    sorted(nuevos_test)
)

,conjunto,equipos,equipos_no_vistos_train
0,Entrenamiento,18,0
1,Validación,13,1
2,Prueba,12,1


Nuevos en validación: ['Costa del Este Academia']
Nuevos en prueba: ['Unión Coclé']


# Celda 15. Verificación final de la matriz de entrada mediante aserciones 
# 6.2. Verificación final de las variables de entrada
Antes de almacenar los subconjuntos verifico nuevamente que ninguna variable correspondiente al resultado final del partido forme parte de las matrices de entrada.Mantengo los goles y el resultado exclusivamente como objetivos o información de evaluación.En esta celda ejecuto un control de seguridad automatizado mediante assert sobre la lista consolidada columnas_X. Al evaluar que la intersección entre las variables prohibidas (columnas_prohibidas) y las columnas seleccionadas para quesea un conjunto vacío (assert len(fugas) == 0), garantizo el aislamiento absoluto de los objetivos y elimino cualquier riesgo de target leakage antes del entrenamiento de los modelos.

In [18]:
# definir el conjunto estricto de variables objetivo prohibidas en X
columnas_prohibidas = {
    "resultado",
    "home_team_goal_count",
    "away_team_goal_count",
    "total_goal_count"
}

# calcular la interseccion entre las variables prohibidas y la lista de predictores X
fugas = (
    columnas_prohibidas
    .intersection(
        set(columnas_X)
    )
)

# mostrar el resultado del control de fuga
print(
    "Variables de resultado presentes en X:",
    fugas
)

# validar formalmente que la interseccion sea completamente vacia
assert len(fugas) == 0

Variables de resultado presentes en X: set()


# Celda 16. Verificación de congruencia estructural en las matrices de entrada 
# 6.3. Verificación de alineación de columnas entre particiones
Antes de alimentar los modelos de aprendizaje automático y los pipelines de preprocesamiento, compruebo que la dimensión y el orden exacto de las columnas de características coincidan entre los subconjuntos de entrenamiento, validación y prueba.

In [19]:
# comprobar alineación exacta de columnas entre entrenamiento y validación
print(
    "Mismas columnas Train / Validación:",
    X_train.columns.equals(
        X_validacion.columns
    )
)

# comprobar alineación exacta de columnas entre entrenamiento y prueba
print(
    "Mismas columnas Train / Test:",
    X_train.columns.equals(
        X_test.columns
    )
)

# verificar la cantidad total de columnas predictoras
print(
    "Número de columnas:",
    X_train.shape[1]
)

Mismas columnas Train / Validación: True
Mismas columnas Train / Test: True
Número de columnas: 48


# Celda 17. Exportación de los subconjuntos temporales procesados
# 7. Exportación de los subconjuntos temporales
Una vez validada la separación, exporto los conjuntos de entrenamiento, validación y prueba conservando tanto las características como las variables objetivo. Mantengo estos archivos sin aplicar imputación, escalado ni codificación categórica. Dichas transformaciones serán ajustadas exclusivamente con los datos de entrenamiento dentro de los procedimientos de cada modelo.

En esta celda defino las rutas correspondientes (lpf_train.csv, lpf_validacion.csv y lpf_test.csv) dentro de PROCESSED_DIR y persisto cada uno de los DataFrames en formato CSV sin índices (index=False) y con codificación utf-8-sig. Esta separación garantiza que la etapa de preprocesamiento y entrenamiento posterior cargue los subconjuntos completamente aislados.

In [20]:
# definir las rutas de destino para cada subconjunto en PROCESSED_DIR
ARCHIVO_TRAIN = (
    PROCESSED_DIR
    / "lpf_train.csv"
)

ARCHIVO_VALIDACION = (
    PROCESSED_DIR
    / "lpf_validacion.csv"
)

ARCHIVO_TEST = (
    PROCESSED_DIR
    / "lpf_test.csv"
)

# exportar las tres particiones temporales a disco en formato CSV
df_train.to_csv(
    ARCHIVO_TRAIN,
    index=False,
    encoding="utf-8-sig"
)

df_validacion.to_csv(
    ARCHIVO_VALIDACION,
    index=False,
    encoding="utf-8-sig"
)

df_test.to_csv(
    ARCHIVO_TEST,
    index=False,
    encoding="utf-8-sig"
)

# mostrar confirmacion de las rutas de los archivos generados
print(
    "Entrenamiento:",
    ARCHIVO_TRAIN
)

print(
    "Validación:",
    ARCHIVO_VALIDACION
)

print(
    "Prueba:",
    ARCHIVO_TEST
)

Entrenamiento: C:\Users\ADMIN003\OneDrive - Universidad Tecnológica de Panamá\Escritorio\II SEMESTRE 2026\TESIS\Sistema_Predictivo_LPF\data\processed\lpf_train.csv
Validación: C:\Users\ADMIN003\OneDrive - Universidad Tecnológica de Panamá\Escritorio\II SEMESTRE 2026\TESIS\Sistema_Predictivo_LPF\data\processed\lpf_validacion.csv
Prueba: C:\Users\ADMIN003\OneDrive - Universidad Tecnológica de Panamá\Escritorio\II SEMESTRE 2026\TESIS\Sistema_Predictivo_LPF\data\processed\lpf_test.csv


# Celda 18 Verificación de archivos exportados

In [21]:
archivos_split = {
    "Entrenamiento": ARCHIVO_TRAIN,
    "Validación": ARCHIVO_VALIDACION,
    "Prueba": ARCHIVO_TEST
}

for nombre, ruta in archivos_split.items():
    df_tmp = pd.read_csv(
        ruta,
        encoding="utf-8-sig"
    )

    print(
        nombre,
        "->",
        df_tmp.shape,
        "| existe:",
        ruta.exists()
    )

Entrenamiento -> (1869, 55) | existe: True
Validación -> (206, 55) | existe: True
Prueba -> (151, 55) | existe: True


# Celda 19  Resumen final
## 8. Resumen de la construcción de los conjuntos
Finalizo la etapa con 1,869 partidos destinados al entrenamiento, 206 encuentros para validación y 151 registros reservados para la evaluación final.
La separación mantiene un orden estrictamente temporal y no existen partidos compartidos entre los tres subconjuntos. También identifico la aparición de clubes no observados previamente, situación que conservaré para evaluar la capacidad de generalización de los modelos.

No realizo ninguna transformación aprendida sobre el conjunto completo. La imputación de datos faltantes, la codificación de variables categóricas y el escalado se ajustarán posteriormente utilizando únicamente los registros de entrenamiento.

In [22]:
resumen_final_split = pd.DataFrame({
    "Indicador": [
        "Partidos de entrenamiento",
        "Partidos de validación",
        "Partidos de prueba",
        "Variables predictoras",
        "Variables numéricas",
        "Variables categóricas",
        "Baseline mayoritario train (%)",
        "Baseline mayoritario validación (%)",
        "Baseline mayoritario prueba (%)"
    ],

    "Valor": [
        len(df_train),
        len(df_validacion),
        len(df_test),
        len(columnas_X),
        len(columnas_numericas),
        len(columnas_categoricas),

        round(
            y_train.eq(
                clase_baseline
            ).mean() * 100,
            2
        ),

        round(
            y_validacion.eq(
                clase_baseline
            ).mean() * 100,
            2
        ),

        round(
            y_test.eq(
                clase_baseline
            ).mean() * 100,
            2
        )
    ]
})

display(resumen_final_split)

,Indicador,Valor
0,Partidos de entrenamiento,1869.00
1,Partidos de validación,206.00
2,Partidos de prueba,151.00
3,Variables predictoras,48.00
4,Variables numéricas,44.00
5,Variables categóricas,4.00
6,Baseline mayoritario train (%),39.06
7,Baseline mayoritario validación (%),36.41
8,Baseline mayoritario prueba (%),37.09
